# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The queue uses the validated held-out predictions from the best-performing model to prioritize items for human review. Items are ranked by the model's probability of being in the declining class, with higher scores reviewed first.

Reason codes are based on observable performance and content signals in the available data. A `recent_decline_signal` indicates a measured drop in recent impressions compared with the previous 30-day period. A `visibility_signal` indicates that search visibility or position is available for review. A `freshness_signal` identifies items where content freshness may warrant review. Items without a clear signal are marked as `mixed_or_uncertain_signal` and require manual diagnosis.

These reason codes are decision-support signals, not causal claims. A high model probability does not prove that refreshing or changing a page will improve its future performance.

In [1]:
# Section 1: Load the processed feature vector from the earlier ML pipeline

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

processed_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "refresh_feature_vector.csv"
)

print("Project root:", PROJECT_ROOT)
print("Processed feature vector:", processed_path)
print("File exists:", os.path.exists(processed_path))

if not os.path.exists(processed_path):
    raise FileNotFoundError(
        f"Processed feature vector not found:\n{processed_path}"
    )

feature_df = pd.read_csv(processed_path)

print("Rows:", len(feature_df))
print("Columns:", len(feature_df.columns))
print("Has target:", "is_declining_label" in feature_df.columns)

if "is_declining_label" not in feature_df.columns:
    raise ValueError(
        "The processed feature vector does not contain is_declining_label."
    )

print("\nTarget distribution:")
print(feature_df["is_declining_label"].value_counts(dropna=False))

Project root: C:\Users\rajku\ML internship notebook\flyrank-ml-internship-starter-main\flyrank-ml-internship-starter-main
Processed feature vector: C:\Users\rajku\ML internship notebook\flyrank-ml-internship-starter-main\flyrank-ml-internship-starter-main\data\processed\refresh_feature_vector.csv
File exists: True
Rows: 30000
Columns: 52
Has target: True

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [2]:
# Inspect processed data and available model-related columns/files
# before generating the ranked action queue.

print("Processed feature columns:")
print(feature_df.columns.tolist())

print("\nModel-related files in the project:")

model_dirs = [
    os.path.join(PROJECT_ROOT, "models"),
    os.path.join(PROJECT_ROOT, "work", "models"),
    os.path.join(PROJECT_ROOT, "data", "processed"),
    os.path.join(PROJECT_ROOT, "work", "outputs"),
]

for folder in model_dirs:
    if os.path.exists(folder):
        print(f"\n[{folder}]")
        for name in os.listdir(folder):
            print(" -", name)

Processed feature columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']

Model-related files in the project:


In [3]:
# Section 1: Inspect the existing model predictions

predictions_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "model_predictions.csv"
)

print("Model predictions path:", predictions_path)
print("File exists:", os.path.exists(predictions_path))

if not os.path.exists(predictions_path):
    raise FileNotFoundError(
        f"Model predictions file not found:\n{predictions_path}"
    )

predictions_df = pd.read_csv(predictions_path)

print("Rows:", len(predictions_df))
print("Columns:", len(predictions_df.columns))

print("\nPrediction columns:")
print(predictions_df.columns.tolist())

print("\nFirst 5 rows:")
display(predictions_df.head())

Model predictions path: C:\Users\rajku\ML internship notebook\flyrank-ml-internship-starter-main\flyrank-ml-internship-starter-main\data\processed\model_predictions.csv
File exists: True
Rows: 30000
Columns: 9

Prediction columns:
['content_id', 'client_id', 'is_declining_label', 'split', 'prob_logistic_regression', 'prob_decision_tree', 'prob_random_forest', 'best_model_name', 'best_model_probability']

First 5 rows:


,content_id,client_id,is_declining_label,split,prob_logistic_regression,prob_decision_tree,prob_random_forest,best_model_name,best_model_probability
0,content_304f48230142,client_f369cb89fc,1,train,0.490173,0.413181,0.542529,random_forest,0.542529
1,content_a1fb4e703a9e,client_4e07408562,1,train,0.632719,0.448492,0.432100,random_forest,0.432100
2,content_9aa793d4d895,client_7f2253d7e2,1,train,0.705139,0.694024,0.814868,random_forest,0.814868
3,content_331d6c4de07b,client_19581e27de,0,train,0.274520,0.316977,0.348316,random_forest,0.348316
4,content_d99b7a2d90ca,client_3fdba35f04,1,train,0.379525,0.694024,0.683803,random_forest,0.683803


In [4]:
# Section 1: Build the ranked review queue from held-out predictions

# Use only held-out rows for the validated review queue.
heldout_predictions = predictions_df[
    predictions_df["split"].astype(str).str.lower().isin(
        ["test", "validation", "holdout"]
    )
].copy()

print("Held-out rows:", len(heldout_predictions))
print("Held-out clients:", heldout_predictions["client_id"].nunique())

if len(heldout_predictions) == 0:
    raise ValueError(
        "No held-out rows were found. Check the values in the 'split' column "
        "before building the ranked queue."
    )

# Rank by the probability produced by the selected best model.
heldout_predictions = heldout_predictions.sort_values(
    "best_model_probability",
    ascending=False
).reset_index(drop=True)

heldout_predictions["rank"] = np.arange(
    1, len(heldout_predictions) + 1
)

print("\nBest model distribution:")
print(heldout_predictions["best_model_name"].value_counts())

print("\nTop 20 ranked items:")
display(
    heldout_predictions[
        [
            "rank",
            "content_id",
            "client_id",
            "best_model_name",
            "best_model_probability",
            "is_declining_label",
        ]
    ].head(20)
)

Held-out rows: 2325
Held-out clients: 6

Best model distribution:
best_model_name
random_forest    2325
Name: count, dtype: int64

Top 20 ranked items:


,rank,content_id,client_id,best_model_name,best_model_probability,is_declining_label
0,1,content_0cf67ec37ab8,client_f74efabef1,random_forest,0.793898,1
1,2,content_6e17dbac0491,client_f74efabef1,random_forest,0.768096,1
2,3,content_575fd096bff5,client_f74efabef1,random_forest,0.761912,1
3,4,content_52b1c884e871,client_f74efabef1,random_forest,0.753956,1
4,5,content_998f6f88784c,client_f74efabef1,random_forest,0.748972,1
5,6,content_6e792cf3ce56,client_f74efabef1,random_forest,0.748845,1
6,7,content_83be0494c955,client_f74efabef1,random_forest,0.746253,1
7,8,content_331182ca4cae,client_f74efabef1,random_forest,0.746120,0
8,9,content_b0284fd0f633,client_f74efabef1,random_forest,0.745993,1
9,10,content_eb53dc14317a,client_f74efabef1,random_forest,0.745270,1


In [5]:
# Section 1: Join ranked predictions with the original feature signals

ranked_queue = heldout_predictions.merge(
    feature_df,
    on=["content_id", "client_id", "is_declining_label"],
    how="left",
    suffixes=("", "_feature")
)

print("Ranked queue rows:", len(ranked_queue))
print("Ranked queue columns:", len(ranked_queue.columns))

# Check that the important review signals are available.
review_columns = [
    "rank",
    "content_id",
    "client_id",
    "best_model_name",
    "best_model_probability",
    "is_declining_label",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "impression_tier",
    "position_tier",
    "freshness_tier",
]

available_review_columns = [
    c for c in review_columns if c in ranked_queue.columns
]

print("\nAvailable review columns:")
print(available_review_columns)

display(ranked_queue[available_review_columns].head(10))

Ranked queue rows: 2325
Ranked queue columns: 59

Available review columns:
['rank', 'content_id', 'client_id', 'best_model_name', 'best_model_probability', 'is_declining_label', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'avg_position', 'impression_tier', 'position_tier', 'freshness_tier']


,rank,content_id,client_id,best_model_name,best_model_probability,is_declining_label,impressions_last_30d,impressions_prev_30d,clicks_last_30d,clicks_prev_30d,sessions_last_30d,sessions_prev_30d,content_age_days,days_since_last_update,avg_position,impression_tier,position_tier,freshness_tier
0,1,content_0cf67ec37ab8,client_f74efabef1,random_forest,0.793898,1,34,312,0,0,2,0,140,8,3.0,moderate,top_3,0-30
1,2,content_6e17dbac0491,client_f74efabef1,random_forest,0.768096,1,24,216,0,0,2,2,140,8,10.6,moderate,striking,0-30
2,3,content_575fd096bff5,client_f74efabef1,random_forest,0.761912,1,57,206,0,0,12,5,140,8,13.9,moderate,striking,0-30
3,4,content_52b1c884e871,client_f74efabef1,random_forest,0.753956,1,75,179,0,0,0,3,140,8,15.1,moderate,striking,0-30
4,5,content_998f6f88784c,client_f74efabef1,random_forest,0.748972,1,227,9661,2,0,7,1,148,8,2.6,good,top_3,0-30
5,6,content_6e792cf3ce56,client_f74efabef1,random_forest,0.748845,1,773,1030,0,1,0,4,148,8,29.1,good,page_3_5,0-30
6,7,content_83be0494c955,client_f74efabef1,random_forest,0.746253,1,25,470,0,0,0,0,140,8,13.7,moderate,striking,0-30
7,8,content_331182ca4cae,client_f74efabef1,random_forest,0.746120,0,1512,919,0,0,19,5,134,20,35.9,good,page_3_5,0-30
8,9,content_b0284fd0f633,client_f74efabef1,random_forest,0.745993,1,108,730,0,3,1,3,117,20,2.8,moderate,top_3,0-30
9,10,content_eb53dc14317a,client_f74efabef1,random_forest,0.745270,1,66,121,0,0,2,1,92,20,7.6,moderate,page_1,0-30


In [6]:
# Section 1: Create practical reason codes and suggested actions
#
# These are review heuristics, not causal rules.
# A reason code explains why an item deserves human attention;
# it does not prove that a particular action will improve performance.

# Work with numeric copies so missing values are handled explicitly.
ranked_queue["impressions_last_30d"] = pd.to_numeric(
    ranked_queue["impressions_last_30d"], errors="coerce"
)
ranked_queue["impressions_prev_30d"] = pd.to_numeric(
    ranked_queue["impressions_prev_30d"], errors="coerce"
)

ranked_queue["clicks_last_30d"] = pd.to_numeric(
    ranked_queue["clicks_last_30d"], errors="coerce"
)
ranked_queue["clicks_prev_30d"] = pd.to_numeric(
    ranked_queue["clicks_prev_30d"], errors="coerce"
)

ranked_queue["sessions_last_30d"] = pd.to_numeric(
    ranked_queue["sessions_last_30d"], errors="coerce"
)
ranked_queue["sessions_prev_30d"] = pd.to_numeric(
    ranked_queue["sessions_prev_30d"], errors="coerce"
)

ranked_queue["content_age_days"] = pd.to_numeric(
    ranked_queue["content_age_days"], errors="coerce"
)

ranked_queue["days_since_last_update"] = pd.to_numeric(
    ranked_queue["days_since_last_update"], errors="coerce"
)

ranked_queue["avg_position"] = pd.to_numeric(
    ranked_queue["avg_position"], errors="coerce"
)

# ---------------------------------------------------------
# 1. Recent-versus-previous performance signal
# ---------------------------------------------------------
# Use only rows where the previous period has measurable impressions.
ranked_queue["impression_change_pct"] = np.where(
    ranked_queue["impressions_prev_30d"] > 0,
    (
        (ranked_queue["impressions_last_30d"]
         - ranked_queue["impressions_prev_30d"])
        / ranked_queue["impressions_prev_30d"]
    ) * 100,
    np.nan
)

# Operational review heuristic:
# a drop of 20% or more is treated as a "recent decline signal".
# This threshold is a prioritization rule, not a causal finding.
ranked_queue["recent_decline_signal"] = (
    ranked_queue["impression_change_pct"] <= -20
)

# ---------------------------------------------------------
# 2. Visibility signal
# ---------------------------------------------------------
ranked_queue["visibility_signal"] = (
    (ranked_queue["impressions_last_30d"] > 0) &
    ranked_queue["position_tier"].notna()
)

# ---------------------------------------------------------
# 3. Freshness signal
# ---------------------------------------------------------
ranked_queue["freshness_signal"] = (
    ranked_queue["freshness_tier"]
    .astype(str)
    .isin(["31-90", "91-180", "181-365", "365+"])
)

# ---------------------------------------------------------
# 4. Assign one primary reason code
# ---------------------------------------------------------
def assign_reason(row):
    if row["recent_decline_signal"]:
        return "recent_decline_signal"
    elif row["freshness_signal"]:
        return "freshness_signal"
    elif row["visibility_signal"]:
        return "visibility_signal"
    else:
        return "mixed_or_uncertain_signal"


ranked_queue["reason_code"] = ranked_queue.apply(
    assign_reason,
    axis=1
)

# ---------------------------------------------------------
# 5. Map reason code to a suggested human action
# ---------------------------------------------------------
action_map = {
    "recent_decline_signal":
        "review_recent_performance_and_consider_refresh",
    "freshness_signal":
        "review_content_freshness",
    "visibility_signal":
        "review_search_visibility_and_position",
    "mixed_or_uncertain_signal":
        "manual_diagnosis_before_action",
}

ranked_queue["suggested_action"] = (
    ranked_queue["reason_code"].map(action_map)
)

# ---------------------------------------------------------
# Show the resulting ranked queue
# ---------------------------------------------------------
queue_columns = [
    "rank",
    "content_id",
    "best_model_probability",
    "reason_code",
    "suggested_action",
    "impression_change_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "position_tier",
    "freshness_tier",
]

display(
    ranked_queue[queue_columns].head(20)
)

print("\nReason-code distribution:")
print(ranked_queue["reason_code"].value_counts())

print("\nSuggested-action distribution:")
print(ranked_queue["suggested_action"].value_counts())

,rank,content_id,best_model_probability,reason_code,suggested_action,impression_change_pct,impressions_last_30d,impressions_prev_30d,content_age_days,days_since_last_update,avg_position,position_tier,freshness_tier
0,1,content_0cf67ec37ab8,0.793898,recent_decline_signal,review_recent_performance_and_consider_refresh,-89.102564,34,312,140,8,3.0,top_3,0-30
1,2,content_6e17dbac0491,0.768096,recent_decline_signal,review_recent_performance_and_consider_refresh,-88.888889,24,216,140,8,10.6,striking,0-30
2,3,content_575fd096bff5,0.761912,recent_decline_signal,review_recent_performance_and_consider_refresh,-72.330097,57,206,140,8,13.9,striking,0-30
3,4,content_52b1c884e871,0.753956,recent_decline_signal,review_recent_performance_and_consider_refresh,-58.100559,75,179,140,8,15.1,striking,0-30
4,5,content_998f6f88784c,0.748972,recent_decline_signal,review_recent_performance_and_consider_refresh,-97.650347,227,9661,148,8,2.6,top_3,0-30
5,6,content_6e792cf3ce56,0.748845,recent_decline_signal,review_recent_performance_and_consider_refresh,-24.951456,773,1030,148,8,29.1,page_3_5,0-30
6,7,content_83be0494c955,0.746253,recent_decline_signal,review_recent_performance_and_consider_refresh,-94.680851,25,470,140,8,13.7,striking,0-30
7,8,content_331182ca4cae,0.746120,visibility_signal,review_search_visibility_and_position,64.526659,1512,919,134,20,35.9,page_3_5,0-30
8,9,content_b0284fd0f633,0.745993,recent_decline_signal,review_recent_performance_and_consider_refresh,-85.205479,108,730,117,20,2.8,top_3,0-30
9,10,content_eb53dc14317a,0.745270,recent_decline_signal,review_recent_performance_and_consider_refresh,-45.454545,66,121,92,20,7.6,page_1,0-30



Reason-code distribution:
reason_code
visibility_signal            1316
recent_decline_signal         911
mixed_or_uncertain_signal      89
freshness_signal                9
Name: count, dtype: int64

Suggested-action distribution:
suggested_action
review_search_visibility_and_position             1316
review_recent_performance_and_consider_refresh     911
manual_diagnosis_before_action                      89
review_content_freshness                             9
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended to help a content or SEO reviewer prioritize which content items deserve attention first. The ranked queue is decision-support: it helps a human choose where to investigate, rather than automatically deciding what should be changed.

The model ranking is based on the validated held-out predictions available in this project. The reason codes add observable performance and content signals to make the queue easier to review.

The results should be used only for prioritization within the data and evaluation setup used here. They should not be treated as a guarantee of future performance, a causal estimate of the effect of refreshing content, or a prediction of any external search engine's algorithm.

The queue is not a production automation system. A human should inspect the content, context, business value, and data quality before taking action. Items with weak or conflicting signals should remain in manual review rather than being assigned an automatic action.

In [7]:
# Section 2: Intended-use checks and limits

print("Intended use: human decision-support for content review prioritization.")
print("Automation level: non-production; human review required.")
print("Ranked items available for review:", len(ranked_queue))

print("\nBest model used in the held-out queue:")
print(ranked_queue["best_model_name"].value_counts())

print("\nModel probability range:")
print(
    "min =",
    round(ranked_queue["best_model_probability"].min(), 4),
    "| max =",
    round(ranked_queue["best_model_probability"].max(), 4)
)

print("\nReason codes requiring manual diagnosis:")
manual_review_count = (
    ranked_queue["reason_code"] == "mixed_or_uncertain_signal"
).sum()

print(manual_review_count)

print("\nLimit checks:")
print("- No causal claim is made from the ranking.")
print("- The queue does not automatically publish, delete, or rewrite content.")
print("- Human review is required before action.")

Intended use: human decision-support for content review prioritization.
Automation level: non-production; human review required.
Ranked items available for review: 2325

Best model used in the held-out queue:
best_model_name
random_forest    2325
Name: count, dtype: int64

Model probability range:
min = 0.0074 | max = 0.7939

Reason codes requiring manual diagnosis:
89

Limit checks:
- No causal claim is made from the ranking.
- The queue does not automatically publish, delete, or rewrite content.
- Human review is required before action.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Every ranked item must be reviewed by a person before any content change is made. The reviewer should first check that the underlying data is meaningful, then inspect the page context, search visibility, recent performance, freshness, and business relevance.

The model score is only a prioritization signal. A high score does not determine the correct intervention. The reviewer should decide whether the appropriate action is to refresh, investigate further, leave the content unchanged, or remove it from the queue.

The following should not be automated by this playbook:

- Automatically rewriting or publishing content.
- Automatically deleting or redirecting pages.
- Automatically changing titles, keywords, or other page elements.
- Automatically treating a model score as proof that a refresh will improve performance.
- Automatically acting on items with missing, conflicting, or clearly unreliable signals.
- Automatically making business or editorial decisions without human review.

The playbook is intended for practical decision-support, not autonomous production decisions.

In [8]:
# Section 3: Human-review checks and no-go cases

# Items that require extra manual diagnosis.
manual_diagnosis = ranked_queue[
    ranked_queue["reason_code"] == "mixed_or_uncertain_signal"
].copy()

print("Items requiring manual diagnosis:", len(manual_diagnosis))

# Check for missing signals that should prevent automatic action.
critical_review_columns = [
    "best_model_probability",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
]

missing_signal_counts = ranked_queue[critical_review_columns].isna().sum()

print("\nMissing critical review signals:")
display(missing_signal_counts)

# Explicit no-go policy for this playbook.
no_go_actions = [
    "auto_publish",
    "auto_rewrite",
    "auto_delete",
    "auto_redirect",
    "auto_change_metadata",
    "auto_business_decision",
]

print("\nNo-go actions:")
for action in no_go_actions:
    print("-", action)

print("\nHuman review requirement: ACTIVE")
print("Automatic content changes: NOT ALLOWED")

Items requiring manual diagnosis: 89

Missing critical review signals:


best_model_probability    0
impressions_last_30d      0
impressions_prev_30d      0
content_age_days          0
days_since_last_update    0
avg_position              0
dtype: int64


No-go actions:
- auto_publish
- auto_rewrite
- auto_delete
- auto_redirect
- auto_change_metadata
- auto_business_decision

Human review requirement: ACTIVE
Automatic content changes: NOT ALLOWED


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The recommendations should be treated as potentially stale when the underlying data changes, model ranking quality weakens, or the relationship between recent performance signals and the declining label changes.

Monitoring should focus on whether the queue remains useful for prioritization rather than assuming that the model will remain valid indefinitely.

Retraining or investigation should be considered when:

- The distribution of model probabilities changes substantially.
- The share of high-priority items changes materially.
- Recent validation performance falls below the previously measured level.
- The relationship between the reason-code signals and observed outcomes changes.
- Important input fields develop substantially more missing values.
- The content population or data collection process changes materially.

These are monitoring and review triggers, not automatic retraining rules. Any retraining should use a fresh validation design and should be checked before the updated model is used for decision-support.

In [9]:
# Section 4: Monitoring checks

print("Monitoring baseline for the current held-out queue")
print("-" * 50)

# Model probability distribution
probability_summary = ranked_queue["best_model_probability"].describe()

print("\nModel probability summary:")
display(probability_summary.to_frame("value"))

# High-priority share
high_priority_threshold = 0.70

high_priority_count = (
    ranked_queue["best_model_probability"] >= high_priority_threshold
).sum()

high_priority_share = high_priority_count / len(ranked_queue)

print("\nHigh-priority threshold:", high_priority_threshold)
print("High-priority items:", high_priority_count)
print(
    "High-priority share:",
    round(high_priority_share * 100, 2),
    "%"
)

# Reason-code distribution
print("\nCurrent reason-code distribution:")
display(
    ranked_queue["reason_code"]
    .value_counts()
    .to_frame("count")
)

# Missingness check for important review fields
monitoring_columns = [
    "best_model_probability",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
]

missing_rates = (
    ranked_queue[monitoring_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nMissing-value rates:")
display(missing_rates.to_frame("missing_rate"))

print("\nMonitoring policy:")
print("- Recheck the queue when data distributions or collection rules change.")
print("- Revalidate model performance before relying on a retrained model.")
print("- Do not automatically retrain or deploy from these checks.")

Monitoring baseline for the current held-out queue
--------------------------------------------------

Model probability summary:


,value
count,2325.000000
mean,0.390629
std,0.229171
min,0.007421
25%,0.202220
50%,0.403197
75%,0.611456
max,0.793898



High-priority threshold: 0.7
High-priority items: 153
High-priority share: 6.58 %

Current reason-code distribution:


,count
reason_code,
visibility_signal,1316
recent_decline_signal,911
mixed_or_uncertain_signal,89
freshness_signal,9



Missing-value rates:


,missing_rate
best_model_probability,0.0
impressions_last_30d,0.0
impressions_prev_30d,0.0
clicks_last_30d,0.0
clicks_prev_30d,0.0
sessions_last_30d,0.0
sessions_prev_30d,0.0
content_age_days,0.0
days_since_last_update,0.0
avg_position,0.0



Monitoring policy:
- Recheck the queue when data distributions or collection rules change.
- Revalidate model performance before relying on a retrained model.
- Do not automatically retrain or deploy from these checks.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked queue is exported to `work/outputs/` so that the research paper can reuse the same decision-support output without rebuilding the analysis manually.

The exported queue contains the model ranking, reason codes, suggested actions, and the main review signals used to support prioritization.

The queue CSV is an output artifact and is intentionally not committed to git. The notebook can regenerate it when the analysis is rerun. Any reusable figures should be saved separately under `work/figures/`.

In [10]:
# Section 5: Export the ranked queue for the paper

outputs_dir = os.path.join(PROJECT_ROOT, "work", "outputs")
os.makedirs(outputs_dir, exist_ok=True)

queue_output_path = os.path.join(
    outputs_dir,
    "w07_ranked_action_queue.csv"
)

# Select the fields needed by the paper.
export_columns = [
    "rank",
    "content_id",
    "client_id",
    "best_model_name",
    "best_model_probability",
    "reason_code",
    "suggested_action",
    "impression_change_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "position_tier",
    "freshness_tier",
]

export_columns = [
    c for c in export_columns
    if c in ranked_queue.columns
]

ranked_queue[export_columns].to_csv(
    queue_output_path,
    index=False
)

print("Queue exported successfully.")
print("Path:", queue_output_path)
print("Rows:", len(ranked_queue))
print("Columns exported:", len(export_columns))

print("\nExport exists:", os.path.exists(queue_output_path))

print("\nExport preview:")
display(
    pd.read_csv(queue_output_path).head(10)
)

Queue exported successfully.
Path: C:\Users\rajku\ML internship notebook\flyrank-ml-internship-starter-main\flyrank-ml-internship-starter-main\work\outputs\w07_ranked_action_queue.csv
Rows: 2325
Columns exported: 19

Export exists: True

Export preview:


,rank,content_id,client_id,best_model_name,best_model_probability,reason_code,suggested_action,impression_change_pct,impressions_last_30d,impressions_prev_30d,clicks_last_30d,clicks_prev_30d,sessions_last_30d,sessions_prev_30d,content_age_days,days_since_last_update,avg_position,position_tier,freshness_tier
0,1,content_0cf67ec37ab8,client_f74efabef1,random_forest,0.793898,recent_decline_signal,review_recent_performance_and_consider_refresh,-89.102564,34,312,0,0,2,0,140,8,3.0,top_3,0-30
1,2,content_6e17dbac0491,client_f74efabef1,random_forest,0.768096,recent_decline_signal,review_recent_performance_and_consider_refresh,-88.888889,24,216,0,0,2,2,140,8,10.6,striking,0-30
2,3,content_575fd096bff5,client_f74efabef1,random_forest,0.761912,recent_decline_signal,review_recent_performance_and_consider_refresh,-72.330097,57,206,0,0,12,5,140,8,13.9,striking,0-30
3,4,content_52b1c884e871,client_f74efabef1,random_forest,0.753956,recent_decline_signal,review_recent_performance_and_consider_refresh,-58.100559,75,179,0,0,0,3,140,8,15.1,striking,0-30
4,5,content_998f6f88784c,client_f74efabef1,random_forest,0.748972,recent_decline_signal,review_recent_performance_and_consider_refresh,-97.650347,227,9661,2,0,7,1,148,8,2.6,top_3,0-30
5,6,content_6e792cf3ce56,client_f74efabef1,random_forest,0.748845,recent_decline_signal,review_recent_performance_and_consider_refresh,-24.951456,773,1030,0,1,0,4,148,8,29.1,page_3_5,0-30
6,7,content_83be0494c955,client_f74efabef1,random_forest,0.746253,recent_decline_signal,review_recent_performance_and_consider_refresh,-94.680851,25,470,0,0,0,0,140,8,13.7,striking,0-30
7,8,content_331182ca4cae,client_f74efabef1,random_forest,0.746120,visibility_signal,review_search_visibility_and_position,64.526659,1512,919,0,0,19,5,134,20,35.9,page_3_5,0-30
8,9,content_b0284fd0f633,client_f74efabef1,random_forest,0.745993,recent_decline_signal,review_recent_performance_and_consider_refresh,-85.205479,108,730,0,3,1,3,117,20,2.8,top_3,0-30
9,10,content_eb53dc14317a,client_f74efabef1,random_forest,0.745270,recent_decline_signal,review_recent_performance_and_consider_refresh,-45.454545,66,121,0,0,2,1,92,20,7.6,page_1,0-30


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.